# 演習① ReAct エージェントの部品を作る（穴埋め形式）

**対応ノートブック**: `notebooks/ch3_01_react_agent.ipynb`
**推奨所要時間**: 約 30 分

`# TODO:` と書かれた箇所を自分で埋めてください。
各課題の最後には **自己チェック**（`assert`）が入っています。`✅ OK` が出れば正解です。

> **始める前に**: `ch3_01_react_agent.ipynb` のカーネルが動いたままだと、モデルが 2 つ分メモリに載ります。
> そちらのタブで **Kernel → Shut Down Kernel** を選んでから始めてください。

---

## セットアップ（完成済み）

In [ ]:
# 実行時間: 数秒
import os
os.environ.setdefault('ANONYMIZED_TELEMETRY', 'False')

import sys
import json
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 第3章の共通部品（chapter3/agent_kit.py）を読み込めるようにする
CH3_DIR = (Path('..') if Path('../agent_kit.py').exists() else Path('chapter3')).resolve()
sys.path.insert(0, str(CH3_DIR))
from agent_kit import LocalChatModel, TraceHandler, parse_tool_calls, run_python as _run_python, workspace_path

from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_core.utils.function_calling import convert_to_openai_tool
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

print('キャッシュ先:', os.environ.get('HF_HOME', '~/.cache/huggingface'))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'使用デバイス: {device}')

# エージェントがファイルを読み書きする作業ディレクトリ（chapter3/workspace）
WORKSPACE = CH3_DIR / 'workspace'
WORKSPACE.mkdir(exist_ok=True)
print(f'作業ディレクトリ: {WORKSPACE}')

In [ ]:
# 実行時間: 約2〜3分
MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if device == 'cuda':
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map='auto')
else:
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32)
model.eval()
print(f'{MODEL_NAME} を読み込みました（モデルのメモリ: {model.get_memory_footprint() / 1024**3:.1f} GB）')

---

## 課題 1: ツールを作る

文章の文字数を数えるツール `count_chars` を作ってください。

- 引数は `text: str`、戻り値は `int`
- 空白（スペース・改行）は数えない
- docstring に「何をするツールか」と「いつ使うか」を書く（これが LLM への説明書になる）

In [ ]:
# TODO: @tool を付けて count_chars を定義してください


schema = convert_to_openai_tool(count_chars)
print(json.dumps(schema, ensure_ascii=False, indent=2))

# ── 自己チェック ──
fn = schema['function']
assert fn['name'] == 'count_chars'
assert len(fn['description']) >= 10, 'docstring に、何をするツールかを書いてください'
assert fn['parameters']['properties']['text']['type'] == 'string'
assert count_chars.invoke({'text': 'こんにちは 世界\n'}) == 7, '空白と改行は数えません'
print('✅ OK')

---

## 課題 2: モデルの出力からツール呼び出しを取り出す

モデルの出力の文字列から、`<tool_call>` と `</tool_call>` に挟まれた JSON を取り出し、
`(ツール名, 引数の辞書)` のリストを返す関数を作ってください。

- 1 つの出力に、ツール呼び出しが複数あることもある
- `re.findall(r'<tool_call>\s*(.*?)\s*</tool_call>', text, re.S)` で中身の文字列を取り出せる
- JSON は `json.loads()` で辞書にする。キーは `name` と `arguments`

In [ ]:
import re


def extract_tool_calls(text: str) -> list[tuple[str, dict]]:
    # TODO: <tool_call> の中身を取り出し、(name, arguments) のリストを返してください
    return None


sample = ('調べてから計算します。\n'
          '<tool_call>\n{"name": "search_docs", "arguments": {"query": "メモリ 上限"}}\n</tool_call>\n'
          '<tool_call>\n{"name": "run_python", "arguments": {"code": "print(2 ** 10)"}}\n</tool_call>')
print(extract_tool_calls(sample))

# ── 自己チェック ──
assert extract_tool_calls(sample) == [('search_docs', {'query': 'メモリ 上限'}), ('run_python', {'code': 'print(2 ** 10)'})]
assert extract_tool_calls('ツールは使いません。答えは 42 です。') == []
print('✅ OK')

---

## 課題 3: ReAct のグラフを組み立てる

`count_chars` と、Python を実行する `python_exec` の 2 つのツールを持つエージェントのグラフを組み立ててください。

- ノード: `agent`（`agent_node`）と `tools`（`ToolNode(tools)`）
- エッジ: `START → agent`、`agent` から `tools_condition` による分かれ道、`tools → agent`

In [ ]:
@tool
def python_exec(code: str) -> str:
    """Python のコードを実行し、print() で表示された内容を返す。計算をするときに使う。"""
    return _run_python(code, WORKSPACE)


tools = [count_chars, python_exec]
llm_with_tools = LocalChatModel(model=model, tokenizer=tokenizer).bind_tools(tools)


def agent_node(state: MessagesState) -> dict:
    return {'messages': [llm_with_tools.invoke(state['messages'])]}


builder = StateGraph(MessagesState)
# TODO: ノードを 2 つ追加してください

# TODO: エッジを 3 つ（うち 1 つは条件つき）追加してください

graph = builder.compile()
print(graph.get_graph().draw_ascii())

# ── 自己チェック ──
g = graph.get_graph()
assert set(g.nodes) == {'__start__', 'agent', 'tools', '__end__'}, set(g.nodes)
edges = {(e.source, e.target, e.conditional) for e in g.edges}
assert ('__start__', 'agent', False) in edges
assert ('tools', 'agent', False) in edges
assert ('agent', 'tools', True) in edges and ('agent', '__end__', True) in edges, 'agent からの分かれ道（tools_condition）がありません'
print('✅ OK')

---

## 課題 4: トレースからコストを集計する

エージェントを 1 回動かし、`TraceHandler` の記録（`trace.records`）から次の 3 つを自分で集計してください。

- `n_llm`: LLM の呼び出し回数（`kind == 'LLM'` の記録の数）
- `n_tool`: ツールの呼び出し回数（`kind == 'ツール'` の記録の数）
- `input_tokens`: LLM の入力トークン数の合計（各記録の `input_tokens`）

In [ ]:
trace = TraceHandler()
result = graph.invoke(
    {'messages': [HumanMessage('「エージェントとは LLM とループとツールである」の文字数を数えてください。')]},
    config={'callbacks': [trace], 'recursion_limit': 8},
)
print(f'回答: {result["messages"][-1].content}\n')

# TODO: trace.records から集計してください
n_llm = None
n_tool = None
input_tokens = None

print(f'LLM {n_llm} 回、ツール {n_tool} 回、入力トークン {input_tokens}')

# ── 自己チェック ──
t = trace.totals()
assert n_llm == t['LLM の呼び出し回数'] and n_llm >= 1
assert n_tool == t['ツールの呼び出し回数']
assert input_tokens == t['入力トークン数']
print('✅ OK')

---

## チャレンジ（任意）

1. 課題 1 の `count_chars` の docstring を「文字を数える」だけに短くしてから、課題 4 を実行し直してください。エージェントはツールを正しく選びましたか？
2. 課題 4 の質問を「この文の文字数を数え、その数の 2 乗を計算してください」に変えると、ツールは何回呼ばれますか？ トレースで確かめてください
3. `trace.records` の `input_tokens` が、LLM を呼ぶたびにどれだけ増えたかを表示してください。何が足されて増えたのでしょうか？